In [1]:
import math
import random
import torch
import torch.nn as nn
import numpy as np 
import pandas as pd
from torch.utils.data import DataLoader,Dataset
path="/kaggle/input/datasets/kaushaltiwari/tiny-shakespeare/tiny-shakespeare.txt"

dataset=open(path)
texts=dataset.read()
vocab=set(texts)
vocab=list(vocab)
tokenizer={v:k for k,v in enumerate(vocab)}
decoder={k:v for k,v in enumerate(vocab)}
ids = [tokenizer[c] for c in texts]
ids=torch.tensor(ids,dtype=torch.long)  
print(ids)
print(len(ids)==len(texts))
training_ids=ids[:1003854]
test_ids=ids[1003854:]
print(len(training_ids)+len(test_ids))
block_len=64
batch_size=32
start=1200
x=training_ids[start:start+block_len]
y=training_ids[start+1:start+block_len+1]
x_batches=[]
y_batches=[]
def get_batch(traininig_ids,block_len,batch_size):
    for i in range(batch_size):
        ran=random.randint(0,len(training_ids)-block_len-1)
        x_batch=training_ids[ran:ran+block_len]
        y_batch=training_ids[ran+1:ran+1+block_len]
        x_batches.append(x_batch)
        y_batches.append(y_batch)
        x=torch.stack(x_batches,dim=0)
        y=torch.stack(y_batches,dim=0)
    return x[0][1:],y[0][:-1]

get_batch(training_ids,block_len,batch_size)
vocab_size=len(vocab)
num_list=[i for i  in range(64)]
class Decoder(nn.Module):
    def __init__(self,vocab_size,block_len,embedding_dim=128,hidden_dim=512):
        super().__init__()
        self.emb=nn.Embedding(vocab_size,embedding_dim)
        self.emb2=nn.Embedding(block_len,embedding_dim)
        self.linear1=nn.Linear(embedding_dim,hidden_dim)
        self.relu=nn.ReLU()
        self.linear2=nn.Linear(hidden_dim,embedding_dim)
        self.normalization=nn.LayerNorm(embedding_dim)
        self.query=nn.Linear(embedding_dim,embedding_dim)
        self.values=nn.Linear(embedding_dim,embedding_dim)
        self.proj=nn.Linear(embedding_dim,embedding_dim)
        self.keys=nn.Linear(embedding_dim,embedding_dim)
        self.normalization2=nn.LayerNorm(embedding_dim)
        self.linear3=nn.Linear(embedding_dim,vocab_size)
    def forward(self,x):
        x=self.emb(x)
        pos=torch.tensor(num_list,dtype=torch.long)
        pos=self.emb2(pos)
        x=x+pos
        res=x
        x=self.normalization(x)
        q=self.query(x)
        v=self.values(x)
        k=self.keys(x)
        k=k.transpose(1,2)
        scores=torch.matmul(q,k)
        scaling=scores/(math.sqrt(q.shape[-1]))
        mask=torch.tril(torch.ones(64,64))
        masked_scores=scaling.masked_fill(mask==0,float('-inf'))
        weights=torch.softmax(masked_scores,dim=-1)
        out=torch.matmul(weights,v)
        x=self.proj(out)
        x=res+x
        res=x
        x=self.normalization2(x)
        x=self.linear1(x)
        x=self.relu(x)
        x=self.linear2(x)
        x=x+res
        output=self.linear3(x)
        return output
model=Decoder(vocab_size,block_len,embedding_dim=128,hidden_dim=512)
optimizer=torch.optim.Adam(model.parameters(),lr=0.0001)

tensor([ 9, 14, 10,  ..., 62, 35, 28])
True
1115394
